# Lab 04 — Data Understanding & Exploratory Data Analysis (EDA)
**Tools:** Pandas · NumPy · Matplotlib

## Learning outcomes
By completing this notebook, you will be able to:
- inspect a dataset’s structure and variable types;
- summarize numerical data using descriptive statistics;
- identify missing values and investigate unusual observations;
- explore distributions and relationships using visualizations;
- communicate findings supported by evidence.

> **Instructions:** Complete each `TODO`, run the cells in order, and write brief interpretations. An unusual value is not automatically an error—investigate before deciding how to handle it.


## 1. Load the dataset

The Palmer Penguins dataset contains measurements of penguins from different species and islands. Load the CSV directly from the provided URL.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

url = "https://huggingface.co/datasets/SIH/palmer-penguins/resolve/main/penguins.csv"

# Load the dataset
df = pd.read_csv(url)

# Display the first five rows
display(df.head())

## 2. Understand the dataset

Inspect the dataset before analyzing it.

**Complete the tasks:**
1. Find the number of rows and columns.
2. Inspect column names and data types.
3. Display summary statistics.
4. Count observations in each penguin species.

In [ ]:
# 1. Number of rows and columns
print("Dataset shape (rows, columns):", df.shape)

# 2. Column names and data types
print("\\nColumn names:")
print(df.columns.tolist())
print("\\nData types:")
display(df.dtypes)

# 3. Non-missing counts and data types
print("\\nDataset information:")
df.info()

# 4. Summary statistics for numerical columns
print("\\nNumerical summary:")
display(df.describe())

# 5. Number of observations in each species
print("\\nSpecies counts:")
display(df["species"].value_counts())

**Interpret your results**

- How many observations and variables are in the dataset?
- Which variables are numerical, and which are categorical?
- Are the species represented equally?

Write your answers below.

**Your observations (generated from the dataset):**

The following cell reports the dataset dimensions, variable types, and whether the species counts are balanced.

In [ ]:
# Data-based interpretation for Section 2
n_rows, n_columns = df.shape
numeric_columns = df.select_dtypes(include="number").columns.tolist()
categorical_columns = df.select_dtypes(exclude="number").columns.tolist()
species_counts = df["species"].value_counts()
balanced = species_counts.max() - species_counts.min() <= 1

print(f"The dataset contains {n_rows} observations and {n_columns} variables.")
print("Numerical variables:", ", ".join(numeric_columns))
print("Categorical variables:", ", ".join(categorical_columns))
print("Species counts:", species_counts.to_dict())
print("Species represented equally:", "Yes (approximately)" if balanced else "No")

## 3. Assess data quality

Missing values can affect summaries and visualizations. First, identify where they occur; do not remove rows automatically.

In [ ]:
# 1. Count missing values in each column
missing_counts = df.isna().sum()
display(missing_counts)

# 2. Percentage of missing values in each column
missing_percent = (df.isna().mean() * 100)
display(missing_percent.round(2))

# 3. Display rows that contain at least one missing value
rows_with_missing = df[df.isna().any(axis=1)]
display(rows_with_missing.head())

**Your observations (generated from the dataset):**

The following cell identifies the columns with missing values and explains how missingness affects the available observations.

In [ ]:
# Data-based interpretation for Section 3
columns_with_missing = missing_counts[missing_counts > 0]
if columns_with_missing.empty:
    print("No missing values were found.")
else:
    print("Columns containing missing values:")
    display(pd.DataFrame({
        "missing_count": columns_with_missing,
        "missing_percent": missing_percent[columns_with_missing.index].round(2)
    }))
    print("Missing values reduce the number of usable observations for calculations involving those columns.")
    print("Before removing or filling values, investigate why they are missing and whether the missingness is systematic.")

## 4. Explore numerical variables

Use descriptive statistics to understand the center and spread of a variable. In this section, focus on `body_mass_g`.

In [ ]:
# Select body mass and exclude missing values
body_mass = df["body_mass_g"].dropna()

# 1. Calculate the mean and median using NumPy
mean_mass = np.mean(body_mass)
median_mass = np.median(body_mass)
print("Mean body mass (g):", round(mean_mass, 2))
print("Median body mass (g):", round(median_mass, 2))

# 2. Find the minimum and maximum
print("Minimum body mass (g):", body_mass.min())
print("Maximum body mass (g):", body_mass.max())

# 3. Calculate standard deviation and quartiles
print("Standard deviation (g):", round(np.std(body_mass), 2))
quartiles = np.percentile(body_mass, [25, 50, 75])
print("25th, 50th, and 75th percentiles (g):", quartiles)

# 4. Display the five smallest and five largest body-mass values
print("\\nFive smallest body-mass values:")
display(body_mass.nsmallest(5))
print("\\nFive largest body-mass values:")
display(body_mass.nlargest(5))

In [ ]:
# Data-based interpretation for Section 4
difference = mean_mass - median_mass
print(f"The mean is {abs(difference):.2f} g {'higher' if difference > 0 else 'lower' if difference < 0 else 'equal'} than the median.")
print(f"Body mass ranges from {body_mass.min()} g to {body_mass.max()} g.")
print(f"The standard deviation is {np.std(body_mass):.2f} g.")
print("The smallest and largest observations should be checked in context; an extreme value is not automatically a data-entry error.")

**Your observations (generated from the dataset):**

The following cell compares the mean and median and reports the observed range and standard deviation.

## 5. Visualize distributions

A histogram helps reveal the shape of a numerical distribution. Create a histogram of body mass and label it clearly.

In [ ]:
# Missing values are excluded for this histogram
mass_values = df["body_mass_g"].dropna()

plt.figure(figsize=(8, 4))
plt.hist(mass_values, bins=15, edgecolor="black")
plt.xlabel("Body mass (g)")
plt.ylabel("Frequency")
plt.title("Distribution of Penguin Body Mass")
plt.tight_layout()
plt.show()

**Your interpretation (generated from the dataset):**

The following cell describes the histogram using the mean, median, and skewness as numerical clues. The histogram itself should also be inspected visually.

In [ ]:
# Data-based interpretation for Section 5
skewness = mass_values.skew()
if abs(skewness) < 0.5:
    shape_description = "roughly symmetric"
elif skewness > 0:
    shape_description = "positively (right) skewed"
else:
    shape_description = "negatively (left) skewed"

print(f"The body-mass distribution appears {shape_description} based on its skewness ({skewness:.2f}).")
print(f"The mean is {mass_values.mean():.1f} g and the median is {mass_values.median():.1f} g.")
print("Use the histogram to see where observations are concentrated and whether there are separated peaks or unusual tails.")

## 6. Explore relationships

A scatter plot helps investigate how two numerical variables vary together. Explore flipper length and body mass.

In [ ]:
# Keep only rows where both plotted variables are present
plot_data = df[["flipper_length_mm", "body_mass_g"]].dropna()

plt.figure(figsize=(8, 4))
x_values = plot_data["flipper_length_mm"]
y_values = plot_data["body_mass_g"]
plt.scatter(x_values, y_values, alpha=0.7)
plt.xlabel("Flipper length (mm)")
plt.ylabel("Body mass (g)")
plt.title("Flipper Length vs Body Mass")
plt.tight_layout()
plt.show()

# Correlation describes the strength and direction of a linear relationship.
print("Pearson correlation:", round(plot_data["flipper_length_mm"].corr(
    plot_data["body_mass_g"]), 3))

**Interpret the scatter plot**

- A positive correlation means body mass tends to increase as flipper length increases; a negative correlation means it tends to decrease.
- Look for separate clusters and points far from the main pattern. Different penguin species may form visible clusters.
- Correlation does not prove causation; other factors may influence both measurements.

**Your interpretation (generated from the dataset):**

The following cell reports the direction and strength of the linear association. Use the scatter plot to look for species clusters and unusual points.

In [ ]:
# Data-based interpretation for Section 6
correlation = plot_data["flipper_length_mm"].corr(plot_data["body_mass_g"])
if correlation >= 0.7:
    strength = "strong positive"
elif correlation >= 0.3:
    strength = "moderate positive"
elif correlation > 0:
    strength = "weak positive"
elif correlation <= -0.7:
    strength = "strong negative"
elif correlation <= -0.3:
    strength = "moderate negative"
elif correlation < 0:
    strength = "weak negative"
else:
    strength = "no linear"

print(f"The variables show a {strength} relationship (Pearson r = {correlation:.3f}).")
print("A correlation does not prove causation. Other variables, such as species, may help explain the pattern.")

In [ ]:
# 1. Calculate mean body mass for each species
mean_by_species = df.groupby("species")["body_mass_g"].mean()
display(mean_by_species)

# 2. Calculate median body mass for each species
median_by_species = df.groupby("species")["body_mass_g"].median()
display(median_by_species)

**Your observations (generated from the dataset):**

The following cell identifies the species with the highest and lowest mean body mass and explains limitations of group averages.

**Your observations (generated from the dataset):**

The following cell identifies the species with the highest mean body mass and compares the group means.

In [ ]:
# Data-based interpretation for Section 7
highest_species = mean_by_species.idxmax()
lowest_species = mean_by_species.idxmin()
print(f"{highest_species} has the highest mean body mass ({mean_by_species.max():.1f} g).")
print(f"{lowest_species} has the lowest mean body mass ({mean_by_species.min():.1f} g).")
print("Group averages reveal differences that may be hidden in the overall distribution.")
print("Remember that group sizes may differ and missing body-mass values reduce the sample size for each species.")
print("Mean values alone do not show the full spread or overlap within each species.")

In [ ]:
# Exclude missing body-mass values before plotting
mass_values = df["body_mass_g"].dropna()

plt.figure(figsize=(7, 3))
plt.boxplot(mass_values)
plt.ylabel("Body mass (g)")
plt.title("Body Mass: Spread and Potential Outliers")
plt.tight_layout()
plt.show()

# IQR rule: values below Q1 - 1.5*IQR or above Q3 + 1.5*IQR
q1 = mass_values.quantile(0.25)
q3 = mass_values.quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
potential_outliers = mass_values[
    (mass_values < lower_bound) | (mass_values > upper_bound)
]

print("Lower IQR boundary:", lower_bound)
print("Upper IQR boundary:", upper_bound)
print("Number of potential outliers:", len(potential_outliers))
display(potential_outliers)

**Your observations:**

The box plot marks observations beyond the whiskers as potential outliers. The IQR rule in the code lists values outside Q1 − 1.5 × IQR and Q3 + 1.5 × IQR. These values should be investigated rather than automatically deleted.

**Key principle:** EDA is not just producing statistics and plots. It is using them to understand the data, ask better questions, and make careful, evidence-based decisions.

## 9. EDA conclusion

The box plot and IQR calculation help flag values outside the usual central range. These are **potential** outliers only. Check the original records and relevant penguin context before deciding whether any value is an error or should be removed.

**Overall conclusion:** EDA describes the dataset's structure, quality, distributions, relationships, and group differences. The numerical summaries and plots should be interpreted together; association does not establish causation.